In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis , QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer , make_column_selector,make_column_transformer
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.svm import SVC
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')
os.chdir('E:\Backup Files Abhinav_Satyam\Practical Machine Learning\Sanjay Sir Git Repo\Datasets\Cases\student_performance')

In [2]:
stud = pd.read_csv('student_dataset_10000_rows.csv')
stud

,study_hours,attendance,sleep_hours,internet_usage,assignments_completed,previous_score,exam_score,placement_status
0,7,56,8,7,10,62,100.00,Placed
1,4,69,5,3,8,56,100.00,Placed
2,11,60,7,6,10,45,100.00,Placed
3,8,99,9,8,4,55,90.17,Placed
4,5,52,8,6,8,40,78.82,Placed
...,...,...,...,...,...,...,...,...
9995,2,58,9,7,8,88,69.31,Not Placed
9996,7,98,6,9,4,87,100.00,Placed
9997,10,44,8,5,10,37,95.94,Placed
9998,10,75,7,5,8,52,88.61,Placed


In [3]:
X, y = stud.drop('placement_status', axis = 1), stud['placement_status']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state=26, stratify=y)
 

In [4]:
svm = SVC(kernel = 'linear') # default is   rbf  • (Radial Basis Function / Gaussian kernel)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
accuracy_score(y_test, y_pred)

0.9993333333333333

# SVC for human resource

In [2]:
os.chdir('E:\Backup Files Abhinav_Satyam\Practical Machine Learning\Sanjay Sir Git Repo\Datasets\Cases\human-resources-analytics')


In [3]:
hr = pd.read_csv('HR_comma_sep.csv')
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [4]:
X, y = hr.drop('left', axis = 1), hr['left']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state=26, stratify=y)
 

In [5]:
ohe = OneHotEncoder(sparse_output=False , drop='first').set_output(transform='pandas')
scaler = StandardScaler().set_output(transform='pandas')
cols_trs = make_column_transformer(
    ( ohe, make_column_selector(dtype_include='object')),
    ( 'passthrough', make_column_selector(dtype_include=['int64', 'float64']))
).set_output(transform = 'pandas')

In [27]:
svm = SVC(kernel='linear')
pip = Pipeline([('CT' , cols_trs),('svm',svm)])

pip.fit(X_train, y_train)
y_pred = pip.predict(X_test)
accuracy_score(y_test, y_pred)

0.7781729273171816

## Trying for different C values it will take more time so we will use tqdm which shows us progress bar

In [9]:
cs = np.linspace(0.001, 10,20)
scores  =[]

for c in tqdm(cs):
    svm = SVC(kernel='linear',C = c)
    pip = Pipeline([('CT' , cols_trs),('scaler',scaler),('svm',svm)])
    pip.fit(X_train, y_train)
    y_pred = pip.predict(X_test)
    scores.append([ c, accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores , columns=['c','score'])

df_scores.sort_values(by='score',ascending=False)

100%|██████████| 20/20 [03:58<00:00, 11.92s/it]


,c,score
1,0.527263,0.778395
2,1.053526,0.778173
3,1.579789,0.778173
4,2.106053,0.778173
12,6.316158,0.778173
5,2.632316,0.778173
6,3.158579,0.778173
7,3.684842,0.778173
8,4.211105,0.778173
9,4.737368,0.778173


In [13]:
svm = SVC(kernel='poly')
pip = Pipeline([('CT' , cols_trs),('scaler',scaler),('svm',svm)])

pip.fit(X_train, y_train)
y_pred = pip.predict(X_test)
accuracy_score(y_test, y_pred)

0.9368748610802401

In [9]:
cs = np.linspace(0.001, 10,20)
scores = []
for c in tqdm(cs):
    for d in [2,3,4]:
        svm = SVC(kernel='poly',C = c,degree = d)
        pip = Pipeline([('CT' , cols_trs),('scaler',scaler),('svm',svm)])
        pip.fit(X_train, y_train)
        y_pred = pip.predict(X_test)
        scores.append([ c, d,accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores , columns=['c','degree','score'])

df_scores.sort_values(by='score',ascending=False)

100%|██████████| 20/20 [02:15<00:00,  6.77s/it]


,c,degree,score
56,9.473737,4,0.959102
53,8.947474,4,0.959102
59,10.000000,4,0.958880
50,8.421211,4,0.958435
47,7.894947,4,0.957991
41,6.842421,4,0.957324
44,7.368684,4,0.957102
38,6.316158,4,0.956212
32,5.263632,4,0.955768
35,5.789895,4,0.955768


# Radial Basis Function RBF

In [10]:
svm = SVC(kernel='rbf')  # default is rbf
pip = Pipeline([('CT' , cols_trs),('scaler',scaler),('svm',svm)])

pip.fit(X_train, y_train)
y_pred = pip.predict(X_test)
accuracy_score(y_test, y_pred)

0.9499888864192043

In [6]:
cs = np.linspace(0.001, 10,20) # 
# C is a regularization parameter that controls the trade-off between maximizing the decision margin and minimizing training errors (misclassifications)
gs = np.linspace(0.1, 10,20) # gammas
scores = []
for c in tqdm(cs):
    for g in gs:
        svm = SVC(kernel='rbf',C = c,gamma = g)
        pip = Pipeline([('CT' , cols_trs),('scaler',scaler),('svm',svm)])
        pip.fit(X_train, y_train)
        y_pred = pip.predict(X_test)
        scores.append([ c, g,accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores , columns=['c','gamma','score'])

df_scores.sort_values(by='score',ascending=False)

100%|██████████| 20/20 [1:48:22<00:00, 325.13s/it]


,c,gamma,score
62,1.579789,1.142105,0.977328
63,1.579789,1.663158,0.977328
82,2.106053,1.142105,0.977106
83,2.106053,1.663158,0.976884
122,3.158579,1.142105,0.976884
...,...,...,...
11,0.001000,5.831579,0.762169
19,0.001000,10.000000,0.762169
18,0.001000,9.478947,0.762169
16,0.001000,8.436842,0.762169
